# Step 10 — Does feature gating help, on top of the tuned config?

Notebook 09 tuned every other hyperparameter of v4's mean-pool arm on Covertype and still
recovered less than half the gap to XGBoost (0.9486 -> 0.9569, gap 0.0203 -> 0.0120). Its own
verdict: **"MOSTLY NOT hyperparameters ... real evidence for trying an architectural fix (e.g.
feature gating) here."** `feature_gating.py` is that fix — five switchable modes implementing
`F_t(x) = F_{t-1}(x) + eta * h_t(g_t(x) (elementwise*) x)`, a learned per-feature relevancy
weight applied before each weak learner's first layer (see the module's own docstring for the
full design rationale).

### What this notebook isolates

Only ONE variable changes across the runs below: the gate mode. Every other hyperparameter is
held at notebook 09's exact winning tuned config (loaded from its own saved Optuna study when
available, otherwise the same numbers transcribed from that notebook's printed output — see
below). This is deliberate: if gating helps, we want that attributable to gating, not to a
different, accidentally-better hyperparameter draw. The gate's OWN hyperparameters (init_bias,
hidden_dim for the dynamic nets) are left at their defaults for this first pass — this notebook
asks "does gating help at all," not yet "what's the best gate config."

### The five hypotheses being compared

| gate_mode | what it is |
|---|---|
| `none` | no gating (re-confirms notebook 09's tuned baseline under this notebook's own code path) |
| `global_static` | one learned per-feature weight, shared by every learner, input-independent |
| `global_dynamic` | one small shared network, input-DEPENDENT, same for every learner |
| `per_learner_static` | each learner has its own learned weights over its own columns, input-independent |
| `per_learner_dynamic` | each learner has its own small network over its own columns, input-DEPENDENT |

`global` vs `per_learner` asks whether relevance is a property of the feature or of the
(expert, feature) pair — the second is the thing hard feature bagging cannot express, and is
closest to the predecessor document's original idea. `static` vs `dynamic` asks whether that
matters once, after training, or adaptively per sample.

## 0. Setup

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")
PROJECT_DIR = os.getcwd(); sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np, pandas as pd, torch, optuna
import matplotlib.pyplot as plt, seaborn as sns
from joblib import Parallel, delayed

from benchmarks_rtdl import BENCHMARKS, load_benchmark, published
from training import get_device
from tuning import load_study, run_config
from feature_gating import GATE_MODES

DEVICE = get_device()

# Knobs, overridable from the shell so a headless run needs no edits:
#   SRP_SUBSAMPLE=2000 SRP_FINAL_SEEDS=1 SRP_FINAL_EPOCHS=2 jupyter nbconvert ...
KEY            = os.environ.get("SRP_KEY", "CO")               # CO = Covertype
ARM            = os.environ.get("SRP_ARM", "meanpool")          # notebook 09's tuned arm
FINAL_EPOCHS   = int(os.environ.get("SRP_FINAL_EPOCHS", "60"))  # matches notebook 09's confirm phase
FINAL_PATIENCE = int(os.environ.get("SRP_FINAL_PATIENCE", "8"))
FINAL_SEEDS    = tuple(range(int(os.environ.get("SRP_FINAL_SEEDS", "3"))))
SUBSAMPLE      = int(os.environ.get("SRP_SUBSAMPLE", "0")) or None

THREADS_PER_JOB = 2
N_JOBS = int(os.environ.get("SRP_JOBS", "0")) or (
    3 if DEVICE.type == "cuda" else max(1, min(8, (os.cpu_count() or 2) // THREADS_PER_JOB)))

RUN_DIR = os.path.join(PROJECT_DIR, "runs"); os.makedirs(RUN_DIR, exist_ok=True)
STAMP = time.strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(RUN_DIR, f"10_progress_{STAMP}.log")
_latest = os.path.join(RUN_DIR, "10_progress.log")
if os.path.islink(_latest) or os.path.exists(_latest):
    os.remove(_latest)
os.symlink(os.path.basename(LOG_PATH), _latest)
def log(msg):
    print(msg, flush=True)
    with open(LOG_PATH, "a") as f: f.write(time.strftime("%H:%M:%S ") + msg + "\n")
def show(df, fmt=None, style=None):
    try:
        s = df.style.format(fmt or {}, na_rep="—"); display(style(s) if style else s)
    except (AttributeError, ImportError):
        display(df.round(4))

sns.set_theme(style="whitegrid", context="notebook")
optuna.logging.set_verbosity(optuna.logging.WARNING)
gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only"
log(f"device: {DEVICE} ({gpu}) | dataset={KEY} arm={ARM} | {N_JOBS} workers | "
    f"final: {len(FINAL_SEEDS)} seeds x {FINAL_EPOCHS} epochs per gate mode "
    f"x {len(GATE_MODES)} gate modes")

## 1. Data, and the tuned config notebook 09 found

In [ ]:
data = load_benchmark(KEY)
arrays = data.arrays()
if SUBSAMPLE:
    n = SUBSAMPLE
    arrays = {"Xtr": arrays["Xtr"][:n], "ytr": arrays["ytr"][:n],
             "Xva": arrays["Xva"][:n // 2], "yva": arrays["yva"][:n // 2],
             "Xte": arrays["Xte"][:n], "yte": arrays["yte"][:n]}
    print(f"SMOKE TEST on {SUBSAMPLE} rows -- numbers below are not comparable to notebook 09.")
log(data.summary())

# The exact winning config notebook 09 found for the tuned mean-pool arm (val loss 0.1275,
# test acc 0.9569 +/- 0.0020). Loaded from its own saved Optuna study when the journal file
# is available (v4/.gitignore excludes it from git, but it persists on disk wherever notebook
# 09 actually ran) -- so this is LIVE-VERIFIED, not re-typed, whenever that file is present.
# The fallback is the same numbers TRANSCRIBED from that notebook's own printed cell output,
# used only if the journal is missing (e.g. running this on a fresh checkout of the repo).
STUDY_NAME = f"tune_{KEY}_{ARM}"
STUDY_PATH = os.path.join(RUN_DIR, f"optuna_{KEY}_{ARM}.journal")
FALLBACK_TUNED_PARAMS = dict(num_learners=58, hidden_dim=52, embed_dim=32, depth=2,
                             feature_frac=0.9762263902472026, dropout=0.010604648974993643,
                             lr=0.0006681941342667133, weight_decay=7.06966160448882e-05,
                             head_hidden=52, head_depth=1, embedding_mode="periodic",
                             d_embedding=8, n_frequencies=16, sigma=0.018219809175994004)
try:
    _study = load_study(STUDY_NAME, STUDY_PATH)
    TUNED_PARAMS = dict(_study.best_params)
    log(f"loaded notebook 09's ACTUAL winning params from {STUDY_PATH} (best val loss {_study.best_value:.4f})")
except Exception as e:
    TUNED_PARAMS = dict(FALLBACK_TUNED_PARAMS)
    log(f"could not load notebook 09's study ({e}); using its TRANSCRIBED printed best_params instead")

print("tuned config held fixed while switching the gate on/off:")
for k, v in TUNED_PARAMS.items(): print(f"  {k:16s} {v}")

TUNED_BASELINE, TUNED_BASELINE_SD = 0.9569, 0.0020   # notebook 09's confirmed test acc (no gate)
BASELINE = {"v4 mean-pool + PLR (untuned, notebook 08)": 0.9486,
           "v4 meanpool -- TUNED, notebook 09 (no gate)": TUNED_BASELINE,
           "our XGBoost (tuned)": 0.9689}
print("\nReference points:")
for k, v in BASELINE.items(): print(f"  {k:42s} {v:.4f}")
print(f"  published MLP / XGBoost: {published(KEY,'MLP')} / {published(KEY,'XGBoost',kind='gbdt')}")

## 2. The comparison -- same tuned config, only the gate switched

In [ ]:
GATE_HYPOTHESES = ["none"] + [m for m in GATE_MODES if m != "none"]

t0 = time.time()
rows = []
for mode in GATE_HYPOTHESES:
    tm0 = time.time()
    params = {**TUNED_PARAMS, "gate_mode": mode}
    res = Parallel(n_jobs=min(N_JOBS, len(FINAL_SEEDS)))(
        delayed(run_config)(ARM, params, arrays, seed, task=data.task, output_dim=data.output_dim,
                            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE, threads=THREADS_PER_JOB)
        for seed in FINAL_SEEDS)
    for r in res:
        r["gate_mode"] = mode
    rows.extend(res)
    accs = [r["test_acc"] for r in res]
    sd = np.std(accs, ddof=1) if len(accs) > 1 else 0.0
    log(f"  gate_mode={mode:20s} done in {(time.time()-tm0)/60:.1f} min | "
        f"test acc {np.mean(accs):.4f} +/- {sd:.4f}")

RESULTS = pd.DataFrame(rows)
log(f"all gate modes done in {(time.time()-t0)/60:.1f} min total")
show(RESULTS, {c: "{:.4f}" for c in RESULTS.columns if RESULTS[c].dtype == float})

## 3. Where does each gate mode land?

In [ ]:
SUMMARY = (RESULTS.groupby("gate_mode")["test_acc"].agg(["mean", "std"])
          .rename(columns={"mean": "acc", "std": "sd"}).reindex(GATE_HYPOTHESES))

rows2 = [{"model": "v4 mean-pool + PLR (untuned, notebook 08)", "acc": BASELINE["v4 mean-pool + PLR (untuned, notebook 08)"]},
        {"model": "our XGBoost (tuned)", "acc": BASELINE["our XGBoost (tuned)"]},
        {"model": "MLP (published)", "acc": published(KEY, "MLP")},
        {"model": "XGBoost (published)", "acc": published(KEY, "XGBoost", kind="gbdt")}]
for mode in GATE_HYPOTHESES:
    label = (f"v4 {ARM} tuned + gate={mode}" if mode != "none"
             else "v4 meanpool -- TUNED, notebook 09 (gate=none, re-confirmed here)")
    rows2.append({"model": label, "acc": SUMMARY.loc[mode, "acc"], "sd": SUMMARY.loc[mode, "sd"]})
T = pd.DataFrame(rows2).set_index("model").sort_values("acc", ascending=False)
show(T, {"acc": "{:.4f}", "sd": "{:.4f}"})

fig, ax = plt.subplots(figsize=(9.5, 5.2))
def _color(i):
    if "gate=none" in i and "notebook 09" in i: return "#7b5cff"
    if i.startswith("v4") and "gate=" in i: return "#9c1c47"
    if "XGBoost" in i: return "#ff8a3d"
    return "#9aa0a6"
colors = [_color(i) for i in T.index]
ax.barh(range(len(T)), T["acc"], xerr=T.get("sd", 0).fillna(0), color=colors, capsize=4)
ax.set_yticks(range(len(T))); ax.set_yticklabels(T.index, fontsize=9); ax.invert_yaxis()
ax.set_xlim(T["acc"].min() - 0.02, T["acc"].max() + 0.01)
ax.set_xlabel("test accuracy")
ax.set_title("Covertype -- does gating help on top of a tuned config?\n"
             "(dark red = gated variants, purple = notebook 09's ungated tuned baseline)",
             fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Findings

*(Computed from the run above.)*

In [ ]:
sep = "=" * 92
print(sep); print(f"FINDINGS -- feature gating on top of tuned v4 ({ARM}) on {BENCHMARKS[KEY]['label']}"); print(sep)

none_row = SUMMARY.loc["none"]
none_sd = none_row["sd"] if pd.notna(none_row["sd"]) else 0.0
best_mode = SUMMARY["acc"].idxmax()
best_row = SUMMARY.loc[best_mode]
best_sd = best_row["sd"] if pd.notna(best_row["sd"]) else 0.0

print(f"\n1. Does this notebook's own ungated run match notebook 09's reported result?")
print(f"   notebook 09 reported : {TUNED_BASELINE:.4f} +/- {TUNED_BASELINE_SD:.4f}")
print(f"   this run (gate=none) : {none_row['acc']:.4f} +/- {none_sd:.4f}")
consistent = abs(none_row["acc"] - TUNED_BASELINE) < 2 * max(TUNED_BASELINE_SD, none_sd, 1e-9)
print("   " + ("consistent with notebook 09 (within ~2 seed-noise bands) -- safe to compare gate modes below"
               if consistent else
               "DIVERGES from notebook 09 -- investigate (e.g. fallback params were used) before trusting #2-4"))

print(f"\n2. Which gate mode scores highest, and does it beat the ungated baseline beyond seed noise?")
for mode in GATE_HYPOTHESES:
    if mode == "none":
        continue
    r = SUMMARY.loc[mode]
    r_sd = r["sd"] if pd.notna(r["sd"]) else 0.0
    delta = r["acc"] - none_row["acc"]
    noise = max(r_sd, none_sd, 1e-9)
    tag = "BEATS baseline" if delta > noise else ("WORSE than baseline" if delta < -noise else "within seed noise")
    print(f"   {mode:20s} {r['acc']:.4f} +/- {r_sd:.4f}   delta={delta:+.4f}   [{tag}]")
print(f"   best mode: {best_mode} ({best_row['acc']:.4f})")

xgb = BASELINE["our XGBoost (tuned)"]
gap_notebook09 = xgb - none_row["acc"]
gap_best_gate = xgb - best_row["acc"]
print(f"\n3. Did the best gate mode close more of the gap to XGBoost than tuning alone did?")
print(f"   gap after tuning alone (notebook 09)      : {gap_notebook09:+.4f}")
print(f"   gap after tuning + best gate ({best_mode:20s}) : {gap_best_gate:+.4f}")

print(f"\n4. Verdict")
gate_beats_baseline = (best_row["acc"] - none_row["acc"]) > max(best_sd, none_sd, 1e-9)
if gate_beats_baseline:
    print(f"   Gating HELPS: {best_mode} beats the tuned ungated baseline beyond seed noise.")
    print( "   -> worth a second pass tuning the gate's own knobs (init_bias, hidden_dim) and/or")
    print( "      trying it on the causal-attention arm and other datasets.")
else:
    print( "   Gating does NOT clearly help here: no mode beats the tuned ungated baseline beyond seed noise.")
    print( "   -> the Covertype gap notebook 09 flagged as 'architectural' is not fixed by THIS")
    print( "      gating mechanism at default settings; either the gate's own hyperparameters need")
    print( "      tuning too, or the remaining gap has a different cause than per-learner feature relevance.")
print(sep)

## 5. Notes / next steps

* The gate's OWN hyperparameters (`init_bias`, `hidden_dim` for the dynamic modes) were left at
  their defaults (4.0, 0) for every mode here -- this notebook asks "does gating help at all,"
  not "what's the best gate config." If a mode looks promising but not conclusive, tuning those
  two knobs (via `tuning.py`'s `_gate_config`, which already reads `params["gate_hidden_dim"]`)
  is the natural next step before ruling it out.
* Only `meanpool` on Covertype was tested, for the same reason notebook 09 scoped down: it was
  the cheapest arm and the one dataset with a real, tuning-resistant gap to explain.
* `tuning.py`'s Optuna search space does NOT include `gate_mode` yet by design -- adding it would
  conflate "does gating help" with "what hyperparameters does a gated model want," which is
  exactly the confound this notebook was built to avoid. Add it only after this notebook's
  verdict says gating is worth keeping.